# snapshot

> Flat namespace serializer

In [ ]:
#| default_exp snapshot

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
from paar.snapshot import snapshot, expand
def test_sorted_and_fields():
    rows = snapshot({'b':1, 'a':'hi'})
    assert [r.name for r in rows]==['a','b']
    assert rows[0].type=='str' and rows[0].value=="'hi'" and rows[0].path=='a'
    assert rows[1].type=='int' and rows[1].qualifier=='builtins'
def test_skips_hidden_and_dunder():
    rows = snapshot({'x':1, '_ih':2, '__y':3}, hidden=frozenset({'_ih'}))
    assert [r.name for r in rows]==['x']
def test_shape_populated():
    assert snapshot({'L':[1,2,3]})[0].shape=='3'
def test_container_flag_and_accessor():
    rows = snapshot({'d':{'a':1}, 'n':5})
    d, n = rows[0], rows[1]
    assert d.is_container is True and d.accessor==('d',)
    assert n.is_container is False and n.accessor==('n',)
def test_expand_dict_then_list():
    ns = {'d': {'x': 1, 'y': [10,20]}}
    kids = expand(ns, ('d',))
    assert [k.name for k in kids]==["'x'", "'y'"]
    assert kids[1].is_container and kids[1].accessor==('d',1) and kids[1].path=="d['y']"
    gk = expand(ns, ('d',1))
    assert [g.name for g in gk]==['0','1'] and gk[0].value=='10' and gk[0].path=="d['y'][0]"
def test_expand_truncates():
    kids = expand({'L': list(range(250))}, ('L',))
    assert len(kids)==101 and kids[-1].is_error is False and 'more' in kids[-1].value
def test_expand_non_container_empty():
    assert expand({'n':5}, ('n',))==[]
for t in [test_sorted_and_fields,test_skips_hidden_and_dunder,test_shape_populated,
          test_container_flag_and_accessor,test_expand_dict_then_list,
          test_expand_truncates,test_expand_non_container_empty]: t()

In [ ]:
#| export
from paar.core import VarInfo
from paar.reprs import safe_repr, get_shape, get_dtype
from paar.resolvers import resolve_for

MAX_CHILDREN = 100

def _var_info(name, v, accessor, path):
    "Build a VarInfo for one value at the given accessor/path."
    try:
        return VarInfo(name=name, type=type(v).__name__,
                       qualifier=getattr(type(v), '__module__', '') or '',
                       value=safe_repr(v), shape=get_shape(v), dtype=get_dtype(v),
                       is_container=resolve_for(v) is not None,
                       path=path, accessor=accessor)
    except Exception as e:
        return VarInfo(name=name, type='?', value=f'<error {e}>', is_error=True,
                       path=path, accessor=accessor)

def snapshot(ns, hidden=frozenset()):
    "namespace dict -> sorted list[VarInfo], skipping hidden and dunder names."
    keys = sorted(k for k in ns if k not in hidden and not k.startswith('__'))
    return [_var_info(k, ns[k], (k,), k) for k in keys]

def _walk(ns, accessor):
    "Resolve a positional accessor (name, *idxs) to (object, readable_path). Raises on bad path."
    name, *idxs = accessor
    obj, path = ns[name], name
    for i in idxs:
        r = resolve_for(obj)
        if r is None: raise KeyError(accessor)
        _, step, obj = r.children(obj)[i]
        path += step
    return obj, path

def expand(ns, accessor):
    "Return one level of children of the value at accessor, as VarInfo (capped at MAX_CHILDREN)."
    try:
        obj, path = _walk(ns, tuple(accessor))
    except Exception:
        return []
    r = resolve_for(obj)
    if r is None: return []
    kids = r.children(obj)
    out = [_var_info(nm, child, tuple(accessor)+(i,), path+step)
           for i,(nm, step, child) in enumerate(kids[:MAX_CHILDREN])]
    if len(kids) > MAX_CHILDREN:
        out.append(VarInfo(name='…', type='', value=f'{len(kids)-MAX_CHILDREN} more',
                           path=path, accessor=tuple(accessor)))
    return out

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()